In [5]:
# -*- coding: utf-8 -*-
"""
Reddit Sampling Pipeline
--------------------------------------------------------
Requires the data to be stored in a folder called 'data'.
Used on the subreddits 

r/changemyview
r/AskReddit
r/TrueReddit
r/AmItheAsshole
r/debate
r/PoliticalDiscussion
r/liberal
r/conservative
r/moderatepolitics
r/worldnews
r/socialjustice


This pipeline only looks at the posts itself, the comments have to be matched in a second step. 
threadminer
"""

"\nReddit Sampling Pipeline\n--------------------------------------------------------\nRequires the data to be stored in a folder called 'data'.\nUsed on the subreddits \n\nr/changemyview\nr/AskReddit\nr/TrueReddit\nr/AmItheAsshole\nr/debate\nr/PoliticalDiscussion\nr/liberal\nr/conservative\nr/moderatepolitics\nr/worldnews\nr/socialjustice\n\n\nThis pipeline only looks at the posts itself, the comments have to be matched in a second step. \nthreadminer\n"

In [6]:
from __future__ import annotations
import hashlib
from typing import Iterable, List, Optional, Tuple, Dict, Union
import numpy as np
import pandas as pd
from pathlib import Path
from glob import glob

In [7]:
import json
from pathlib import Path
from typing import Union, Iterable, List, Tuple
import pandas as pd
from glob import glob


# ==========================================================
# Columns to retain from the original Reddit JSONL files
#  - output columns are kept fully and "logically" ordered
# ==========================================================
KEEP_COLS = [
    "author",
    "created_utc",
    "downs",
    "id",
    "likes",
    "num_comments",
    "ups",
    "selftext",
    "title",
    "subreddit",
    "subreddit_id",
    "is_deleted",
    "author_flair_text",
    "distinguished",
    "removed_by_category",
    "removed_by",
    "is_self",
    "locked",
    "archived",
    "stickied",
    "edited",
    "spoiler",
]
KEEP_COLS = list(dict.fromkeys(KEEP_COLS))  # remove duplicates while preserving order


# Logical output ordering: keep "core" first, then moderation/state, then flair/ids/other
OUT_COLS = [
    # Core identifiers / basic metadata
    "id",
    "subreddit",
    "subreddit_id",
    "author",
    "created_utc",
    # Engagement
    "ups",
    "downs",
    "likes",
    "num_comments",
    # Content
    "title",
    "selftext",
    # State / moderation flags
    "is_self",
    "is_deleted",
    "edited",
    "locked",
    "archived",
    "stickied",
    "spoiler",
    "distinguished",
    "removed_by",
    "removed_by_category",
    # Extras
    "author_flair_text",
]
# Ensure OUT_COLS contains *all* KEEP_COLS exactly once (append any missing at the end)
OUT_COLS = list(
    dict.fromkeys([c for c in OUT_COLS if c in KEEP_COLS] + [c for c in KEEP_COLS if c not in OUT_COLS])
)


def load_reddit_jsonl_files(
        paths: Union[str, Path, Iterable[Union[str, Path]]],
        *,
        verbose: bool = True,
        max_error_previews: int = 3,
        drop_title_equals: str = "removed",
        min_num_comments: int = 10,
) -> pd.DataFrame:
    """
    Robust loader for one or multiple Reddit JSONL files.

    - Reads JSON Lines (.jsonl) line-by-line (robust to malformed JSON rows)
    - Keeps only predefined columns (KEEP_COLS)
    - Converts `created_utc` (Unix timestamp) into UTC datetime
    - Ensures consistent schema across all input files
    - Logs skipped malformed lines per file
    - Output columns follow OUT_COLS (which includes ALL KEEP_COLS)

    Additional filtering (applied AFTER concatenation):
    - Drop rows where title == drop_title_equals (case-insensitive, whitespace-trimmed)
    - Drop rows where num_comments < min_num_comments
    - Prints how many rows were dropped by each rule (and total)
    """

    # Convert a single path into a list
    if isinstance(paths, (str, Path)):
        paths = [paths]

    dfs: List[pd.DataFrame] = []
    total_skipped = 0
    processed_files = 0
    failed_files = 0

    for p in paths:
        p = Path(p)
        if not p.exists():
            if verbose:
                print(f"[WARN] File not found and skipped: {p}")
            continue

        rows: List[dict] = []
        skipped = 0
        previews: List[Tuple[int, str]] = []

        try:
            with p.open("r", encoding="utf-8") as f:
                for i, line in enumerate(f, 1):
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        rows.append(json.loads(line))
                    except Exception as e:
                        skipped += 1
                        if len(previews) < max_error_previews:
                            previews.append((i, str(e)))

            processed_files += 1
            total_skipped += skipped

            if verbose and skipped > 0:
                print(f"[WARN] {p.name}: skipped {skipped} malformed lines")
                for ln, msg in previews:
                    print(f"       - line {ln}: {msg}")

            if not rows:
                if verbose:
                    print(f"[WARN] {p.name}: no valid rows found (empty or fully malformed)")
                continue

            df = pd.DataFrame(rows)

            # Ensure missing expected columns exist (filled with NA)
            for c in KEEP_COLS:
                if c not in df.columns:
                    df[c] = pd.NA

            # Select only the relevant columns (stable)
            df = df.loc[:, KEEP_COLS].copy()

            # Guard against duplicate column labels (shouldn't happen after KEEP_COLS dedupe, but defensive)
            if df.columns.duplicated().any():
                df = df.loc[:, ~df.columns.duplicated()].copy()

            # Convert Unix timestamp to UTC datetime (robust to bad types)
            df["created_utc"] = pd.to_numeric(df["created_utc"], errors="coerce")
            df["created_utc"] = pd.to_datetime(df["created_utc"], unit="s", utc=True, errors="coerce")

            # Enforce final column order for this chunk (and ensure all OUT_COLS exist)
            for c in OUT_COLS:
                if c not in df.columns:
                    df[c] = pd.NA
            df = df.loc[:, OUT_COLS].copy()

            dfs.append(df)

        except Exception as e:
            failed_files += 1
            if verbose:
                print(f"[ERROR] Failed to read {p.name}: {e}")
            continue

    # If no file was successfully processed, return an empty DataFrame with the correct schema
    if not dfs:
        if verbose:
            print("[INFO] No data loaded. Returning empty DataFrame.")
        return pd.DataFrame(columns=OUT_COLS)

    # Concatenate all loaded DataFrames (safe because all chunks share OUT_COLS uniquely)
    df_all = pd.concat(dfs, ignore_index=True)

    # Final defensive ordering
    for c in OUT_COLS:
        if c not in df_all.columns:
            df_all[c] = pd.NA
    df_all = df_all.loc[:, OUT_COLS].copy()

    # ==========================================================
    # Post-load filters
    # ==========================================================
    before = len(df_all)

    # 1) Drop title == "deleted" (case-insensitive; also trims whitespace)
    title_norm = df_all["title"].fillna("").astype(str).str.strip().str.lower()
    drop_title_norm = (drop_title_equals or "").strip().lower()
    mask_title_ok = title_norm != drop_title_norm
    dropped_title = int((~mask_title_ok).sum())
    df_all = df_all.loc[mask_title_ok].copy()

    # 2) Drop num_comments < 10 (robust numeric conversion)
    df_all["num_comments"] = pd.to_numeric(df_all["num_comments"], errors="coerce")
    mask_comments_ok = df_all["num_comments"].fillna(-1) >= int(min_num_comments)
    dropped_comments = int((~mask_comments_ok).sum())
    df_all = df_all.loc[mask_comments_ok].copy()

    after = len(df_all)
    dropped_total = before - after

    if verbose:
        print("\n--- Load summary ---")
        print("Processed files:", processed_files)
        print("Failed files:", failed_files)
        print("Total skipped malformed lines:", total_skipped)
        print("Total rows loaded (before filters):", before)

        print("\n--- Filters ---")
        print(f"Dropped because title == '{drop_title_equals}':", dropped_title)
        print(f"Dropped because num_comments < {min_num_comments}:", dropped_comments)
        print("Total dropped by filters:", dropped_total)
        print("Total rows remaining:", after)

    return df_all


# ==========================================
# usage
# ==========================================

files = glob("data/*_posts.jsonl")
df_all = load_reddit_jsonl_files(files, verbose=True)

print(df_all.head())
print("Rows:", len(df_all))




[WARN] r_politics_posts.jsonl: skipped 1 malformed lines
       - line 570: Unterminated string starting at: line 1 column 689 (char 688)

--- Load summary ---
Processed files: 12
Failed files: 0
Total skipped malformed lines: 1
Total rows loaded (before filters): 174899

--- Filters ---
Dropped because title == 'removed': 0
Dropped because num_comments < 10: 133923
Total dropped by filters: 133923
Total rows remaining: 40976
         id subreddit subreddit_id                author  \
1   1dc893q  politics     t5_2cneq  nosotros_road_sodium   
6   1dc8lt8  politics     t5_2cneq  Last_Lonely_Traveler   
16  1dc97qr  politics     t5_2cneq           semafornews   
17  1dc98e9  politics     t5_2cneq              StcStasi   
18  1dc9f9k  politics     t5_2cneq            lotta_love   

                 created_utc   ups  downs likes  num_comments  \
1  2024-06-10 00:08:30+00:00    52      0  None            38   
6  2024-06-10 00:26:39+00:00     0      0  None            13   
16 2024-06-10 

In [8]:
# Optional NLP dependencies
_HAS_SBERT = False
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    _HAS_SBERT = True
except Exception:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity


# ------------------------
# Topics
# ------------------------

DEFAULT_TOPICS = [
    "The government should not forgive student loan debt.",
    "Airbnb should be banned in cities.",
    "The federal minimum wage should be increased.",
    "The US should provide financial and military aid to Ukraine.",
    "A universal basic income would kill the economy.",
    "Climate change is one of the greatest threats to humanity.",
    "Fur clothing should be banned.",
    "The government should not invest in renewable energy.",
    "There should only be vegetarian food in cantines.",
    "Gender-neutral language and stating pronouns are silly issues.",
    "Prostitution should be illegal.",
    "Employers should mandate vaccination.",
    "The government should not be responsible for universal health care.",
    "Immigrants should adopt the local language and culture.",
    "We need stricter gun control laws.",
    "The death penalty should be reestablished.",
    "Police officers should wear body cameras.",
    "Artificial Intelligence should replace humans where possible.",
    "Social media is a threat to democracy.",
]


# ------------------------
# Build topic model
# ------------------------

def build_topic_model(topics: List[str]):
    """
    Build topic embeddings (SBERT) or TF-IDF vectors.
    """
    if _HAS_SBERT:
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        topic_embeddings = model.encode(topics, show_progress_bar=False)
        return "sbert", model, topic_embeddings
    else:
        vec = TfidfVectorizer(min_df=1, max_df=0.95, ngram_range=(1, 2))
        topic_matrix = vec.fit_transform(topics)
        return "tfidf", vec, topic_matrix


# ------------------------
# Core filtering function
# ------------------------

def filter_by_topic_similarity(
        df: pd.DataFrame,
        similarity_threshold: float = 0.4,
        topics: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, int, int]:
    """
    Filter dataframe rows based on cosine similarity between `title`
    and a set of topic sentences.

    Requirements:
    - `df` must contain a column `title`.
    - Rows with title == "[removed]" or "[deleted]" are discarded immediately.

    Returns:
    - filtered_df: rows with similarity >= threshold
    - kept: number of rows retained
    - dropped: number of rows removed
    """

    # Work on a copy
    df = df.copy()

    # FIX: guard against duplicate column labels (this is what triggered your ValueError)
    if df.columns.duplicated().any():
        # keep first occurrence (or rename if you prefer)
        df = df.loc[:, ~df.columns.duplicated()].copy()

    if "title" not in df.columns:
        raise ValueError("Expected column 'title' in dataframe.")

    # Remove [removed] / [deleted] immediately (use .loc, not df[mask])
    mask_valid = ~df["title"].isin(["[removed]", "[deleted]"])
    df = df.loc[mask_valid].copy()

    if df.empty:
        return df, 0, 0

    topics = topics or DEFAULT_TOPICS
    texts = df["title"].fillna("").astype(str).tolist()

    # Build topic model
    kind, model, topic_repr = build_topic_model(topics)

    # Compute similarities
    if kind == "sbert":
        text_embeddings = model.encode(texts, show_progress_bar=False)
        sims = cosine_similarity(text_embeddings, topic_repr)
    else:
        # TF-IDF fallback: fit on topics + texts
        vec = TfidfVectorizer(min_df=1, max_df=0.95, ngram_range=(1, 2))
        combined = topics + texts
        X = vec.fit_transform(combined)
        topic_matrix = X[: len(topics), :]
        text_matrix = X[len(topics):, :]
        sims = cosine_similarity(text_matrix, topic_matrix)

    # Determine best topic per row
    best_idx = sims.argmax(axis=1)
    best_sim = sims.max(axis=1)

    df["best_topic_index"] = best_idx
    df["best_topic"] = [topics[i] for i in best_idx]
    df["best_topic_similarity"] = best_sim

    # Threshold filtering
    keep_mask = df["best_topic_similarity"] >= similarity_threshold
    filtered_df = df.loc[keep_mask].copy()

    kept = int(keep_mask.sum())
    dropped = int(len(df) - kept)

    print(f"Submissions kept: {kept}")
    print(f"Submissions dropped: {dropped}")

    return filtered_df, kept, dropped



# ------------------------
# Example usage
# ------------------------

if __name__ == "__main__":
   
    df_posts = df_all
    print(len(df_posts))

    SIMILARITY_THRESHOLD = 0.4
    filtered, kept, dropped = filter_by_topic_similarity(
        df=df_posts,             # your main dataframe from loader
        similarity_threshold=SIMILARITY_THRESHOLD,
        topics=DEFAULT_TOPICS,
    )

    print(filtered.head())
    print(f"Final kept: {kept}, dropped: {dropped}")


40976
Submissions kept: 998
Submissions dropped: 39978
          id subreddit subreddit_id                author  \
84   1dcetz6  politics     t5_2cneq         Kate_Matthews   
129  1dcjmbw  politics     t5_2cneq       eustachian_lube   
161  1dcnh9q  politics     t5_2cneq         Ok-Story-9319   
175  1dcoil5  politics     t5_2cneq  einsteinfrankenstein   
210  1dcqjgm  politics     t5_2cneq                N0b0me   

                  created_utc   ups  downs likes  num_comments  \
84  2024-06-10 06:19:49+00:00  1071      0  None            96   
129 2024-06-10 11:51:54+00:00     0      0  None            43   
161 2024-06-10 14:55:30+00:00   343      0  None           376   
175 2024-06-10 15:38:23+00:00    14      0  None            17   
210 2024-06-10 17:01:56+00:00    41      0  None            34   

                                                 title  ... archived  \
84   US pushes for $50 billion loan to Ukraine usin...  ...    False   
129               The California Mini

In [9]:
filtered.to_json("sampled.ndjson", orient="records", lines=True, force_ascii=False)

In [10]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


# ------------------------
# Load NDJSON
# ------------------------
def load_ndjson(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"NDJSON not found: {path}")
    df = pd.read_json(path, lines=True)
    if df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()
    return df


# ------------------------
# Table helpers
# ------------------------
def freq_table(
        df: pd.DataFrame,
        col: str,
        top_n: int = 50,
        dropna: bool = True,
) -> pd.DataFrame:
    if col not in df.columns:
        raise ValueError(f"Missing column '{col}'. Available: {list(df.columns)}")

    s = df[col]
    if dropna:
        s = s.dropna()

    counts = s.astype(str).value_counts(dropna=False)
    out = counts.rename("count").to_frame()
    out["share"] = out["count"] / out["count"].sum()
    out = out.reset_index().rename(columns={"index": col})

    # add rank and cut to top_n
    out.insert(0, "rank", np.arange(1, len(out) + 1))
    if top_n is not None:
        out = out.head(top_n)

    return out


def num_comments_summary(df: pd.DataFrame, col: str = "num_comments") -> pd.DataFrame:
    if col not in df.columns:
        raise ValueError(f"Missing column '{col}'. Available: {list(df.columns)}")

    s = pd.to_numeric(df[col], errors="coerce").dropna()
    s = s[s >= 0]

    summary = {
        "n": int(s.shape[0]),
        "mean": float(s.mean()) if len(s) else np.nan,
        "std": float(s.std(ddof=1)) if len(s) > 1 else np.nan,
        "min": float(s.min()) if len(s) else np.nan,
        "p50": float(s.quantile(0.50)) if len(s) else np.nan,
        "p75": float(s.quantile(0.75)) if len(s) else np.nan,
        "p90": float(s.quantile(0.90)) if len(s) else np.nan,
        "p95": float(s.quantile(0.95)) if len(s) else np.nan,
        "p99": float(s.quantile(0.99)) if len(s) else np.nan,
        "max": float(s.max()) if len(s) else np.nan,
    }
    return pd.DataFrame([summary])


def num_comments_bins(
        df: pd.DataFrame,
        col: str = "num_comments",
        bins: str | int | list = "auto",
) -> pd.DataFrame:
    """
    bins:
      - "auto" (numpy heuristic)
      - int number of bins
      - list of bin edges (e.g. [0,1,2,5,10,20,50,100,200,500,1000, np.inf])
    """
    if col not in df.columns:
        raise ValueError(f"Missing column '{col}'. Available: {list(df.columns)}")

    s = pd.to_numeric(df[col], errors="coerce").dropna()
    s = s[s >= 0]

    if isinstance(bins, list):
        cats = pd.cut(s, bins=bins, right=False, include_lowest=True)
    else:
        # numpy bin edges -> turn into intervals with pd.cut
        edges = np.histogram_bin_edges(s, bins=bins)
        cats = pd.cut(s, bins=edges, include_lowest=True)

    counts = cats.value_counts().sort_index()
    out = counts.rename("count").to_frame()
    out["share"] = out["count"] / out["count"].sum()
    out = out.reset_index().rename(columns={"index": "bin"})
    return out


# ------------------------
# Main
# ------------------------
if __name__ == "__main__":
    NDJSON_PATH = "sampled.ndjson"  # <- set your file path
    df = load_ndjson(NDJSON_PATH)
    print(f"Loaded {len(df):,} rows from {NDJSON_PATH}")

    # Columns (adjust if needed)
    SUBREDDIT_COL = "subreddit"
    TOPIC_COL = "best_topic"
    NUM_COMMENTS_COL = "num_comments"

    # 1) Subreddit distribution table
    subreddit_tbl = freq_table(df, SUBREDDIT_COL, top_n=50)
    print("\n=== Subreddit distribution (top 50) ===")
    print(subreddit_tbl.to_string(index=False))

    # 2) Topic distribution table
    topic_tbl = freq_table(df, TOPIC_COL, top_n=50)
    print("\n=== Topic distribution (top 50) ===")
    print(topic_tbl.to_string(index=False))

    # 3a) num_comments summary stats table
    summary_tbl = num_comments_summary(df, NUM_COMMENTS_COL)
    print("\n=== num_comments summary ===")
    print(summary_tbl.to_string(index=False))

    # 3b) num_comments binned distribution table (useful fixed bins)
    fixed_bins = [0, 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, np.inf]
    bins_tbl = num_comments_bins(df, NUM_COMMENTS_COL, bins=fixed_bins)
    print("\n=== num_comments distribution (bins) ===")
    print(bins_tbl.to_string(index=False))


Loaded 998 rows from sampled.ndjson

=== Subreddit distribution (top 50) ===
 rank           subreddit  count    share
    1           AskReddit    586 0.587174
    2           worldnews    199 0.199399
    3        changemyview     67 0.067134
    4        Conservative     65 0.065130
    5       AmItheAsshole     39 0.039078
    6 PoliticalDiscussion     15 0.015030
    7    moderatepolitics     13 0.013026
    8            politics      9 0.009018
    9             Liberal      4 0.004008
   10          TrueReddit      1 0.001002

=== Topic distribution (top 50) ===
 rank                                                          best_topic  count    share
    1        The US should provide financial and military aid to Ukraine.    211 0.211423
    2       Artificial Intelligence should replace humans where possible.    103 0.103206
    3                    A universal basic income would kill the economy.     97 0.097194
    4          Climate change is one of the greatest threats to 